# LeetCode 1037 — Valid Boomerang bằng Python

**Link đề:** https://leetcode.com/problems/valid-boomerang/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry (hình học tọa độ)

```python
class Solution:
    def isBoomerang(self, points: list[list[int]]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu điều kiện "3 điểm **phân biệt** và **không thẳng hàng**".
2. Ôn kiến thức tọa độ: vector, **hệ số góc (slope)**, **diện tích tam giác** (công thức shoelace), **tích có hướng (cross product)**.
3. Ôn công cụ Python: unpacking `(x1, y1), (x2, y2), (x3, y3) = points`, `set` + `tuple`, `fractions.Fraction`, `math.gcd`, vì sao số thực (`float`) nguy hiểm khi so sánh bằng.
4. Viết lời giải từ trực giác (so hệ số góc) → an toàn (nhân chéo) → gọn nhất (tích có hướng ≠ 0).
5. Chứng minh, phân tích độ phức tạp, so sánh các cách; làm bài tập biến thể (n điểm thẳng hàng, hướng quay trái/phải).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng `points` gồm **đúng 3 điểm**, `points[i] = [xi, yi]` trên mặt phẳng. Trả về `True` nếu 3 điểm tạo thành một **boomerang**.

**Boomerang** = 3 điểm **đôi một phân biệt** và **không nằm trên một đường thẳng**.

| `points` | Output | Vì sao |
|---|:-:|---|
| `[[1,1],[2,3],[3,2]]` | `True` | 3 điểm khác nhau, tạo tam giác thật |
| `[[1,1],[2,2],[3,3]]` | `False` | cùng nằm trên đường `y = x` |
| `[[0,0],[0,0],[1,1]]` | `False` | có 2 điểm trùng nhau |
| `[[0,0],[0,2],[0,5]]` | `False` | cùng nằm trên đường thẳng đứng `x = 0` |

**Constraints:** `points.length == 3`, `points[i].length == 2`, `0 <= xi, yi <= 100`.

Gợi ý của đề: *3 điểm là boomerang ⟺ tam giác tạo bởi chúng có **diện tích khác 0**.*

In [1]:
points = [[1, 1], [2, 3], [3, 2]]
(x1, y1), (x2, y2), (x3, y3) = points
print("A =", (x1, y1), "| B =", (x2, y2), "| C =", (x3, y3))

A = (1, 1) | B = (2, 3) | C = (3, 2)


`(x1, y1), (x2, y2), (x3, y3) = points` là **unpacking lồng**: list ngoài có 3 phần tử, mỗi phần tử là list 2 số → tách thẳng ra 6 biến. Nếu `points` không đúng hình dạng sẽ báo `ValueError` — nhờ đề đảm bảo 3 điểm, ta dùng thoải mái.

## 2. Nền tảng hình học tọa độ

### 2.1. Vector giữa hai điểm

Vector `AB = (x2 - x1, y2 - y1)`: "đi từ A tới B phải dịch ngang bao nhiêu, dọc bao nhiêu".

In [2]:
A, B, C = (1, 1), (2, 3), (3, 2)
AB = (B[0] - A[0], B[1] - A[1])
AC = (C[0] - A[0], C[1] - A[1])
print("AB =", AB, "| AC =", AC)

AB = (1, 2) | AC = (2, 1)


### 2.2. Hệ số góc (slope)

Hệ số góc của đường AB là `dy / dx = (y2 - y1) / (x2 - x1)`. Ba điểm thẳng hàng ⟺ AB và AC có cùng hệ số góc.

Vấn đề: đường **thẳng đứng** có `dx = 0` → chia cho 0!

In [3]:
def slope(P, Q):
    return (Q[1] - P[1]) / (Q[0] - P[0])

print("slope AB =", slope(A, B), "| slope AC =", slope(A, C))
try:
    slope((0, 0), (0, 5))
except ZeroDivisionError as e:
    print("Đường thẳng đứng -> ZeroDivisionError:", e)

slope AB = 2.0 | slope AC = 0.5
Đường thẳng đứng -> ZeroDivisionError: division by zero


### 2.3. Số thực `float` và so sánh bằng

`float` lưu xấp xỉ nhị phân → hai phép tính "về lý thuyết bằng nhau" có thể ra kết quả lệch rất nhỏ.

In [4]:
print(0.1 + 0.2, "== 0.3 ?", 0.1 + 0.2 == 0.3)
print(1 / 3 * 3 == 1, 49 / 49 == 1, (1 / 49) * 49 == 1)

0.30000000000000004 == 0.3 ? False
True True False


Kết luận: với tọa độ **nguyên**, hãy tránh phép chia; dùng **nhân chéo** để mọi thứ là số nguyên — so sánh tuyệt đối chính xác.

### 2.4. Nhân chéo thay cho so sánh phân số

`a/b == c/d` (với `b, d ≠ 0`) ⟺ `a·d == c·b`. Và dạng nhân chéo **vẫn có nghĩa** khi `b = 0` hoặc `d = 0`.

Áp dụng: slope AB == slope AC ⟺ `(y2 - y1)·(x3 - x1) == (y3 - y1)·(x2 - x1)`.

### 2.5. Tích có hướng (cross product) 2D và diện tích tam giác

Với hai vector `u = (ux, uy)`, `v = (vx, vy)`: `cross(u, v) = ux·vy - uy·vx`.

- `|cross(AB, AC)|` = diện tích **hình bình hành** dựng trên AB, AC = **2 × diện tích tam giác ABC**.
- `cross = 0` ⟺ AB, AC cùng phương (song song) ⟺ A, B, C thẳng hàng (hoặc có điểm trùng).
- Dấu của `cross` cho biết C ở **bên trái** (`> 0`) hay **bên phải** (`< 0`) của hướng A → B.

In [5]:
def cross(u, v):
    return u[0] * v[1] - u[1] * v[0]

print("cross(AB, AC) =", cross(AB, AC), "-> diện tích tam giác =", abs(cross(AB, AC)) / 2)
print("cross cho 3 điểm thẳng hàng:", cross((1, 1), (2, 2)))
print("cross khi B trùng A (AB = (0,0)):", cross((0, 0), (5, 7)))

cross(AB, AC) = -3 -> diện tích tam giác = 1.5
cross cho 3 điểm thẳng hàng: 0
cross khi B trùng A (AB = (0,0)): 0


### 2.6. Công thức shoelace (dây giày) cho diện tích tam giác

`S = |x1(y2 - y3) + x2(y3 - y1) + x3(y1 - y2)| / 2`

Khai triển ra sẽ thấy biểu thức trong trị tuyệt đối **chính là** `cross(AB, AC)`.

In [6]:
def shoelace_twice_area(p1, p2, p3):
    (x1, y1), (x2, y2), (x3, y3) = p1, p2, p3
    return x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2)

for pts in [((1, 1), (2, 3), (3, 2)), ((0, 0), (4, 0), (0, 3)), ((1, 1), (2, 2), (3, 3))]:
    A_, B_, C_ = pts
    ab = (B_[0] - A_[0], B_[1] - A_[1]); ac = (C_[0] - A_[0], C_[1] - A_[1])
    print(pts, "| shoelace =", shoelace_twice_area(*pts), "| cross =", cross(ab, ac))

((1, 1), (2, 3), (3, 2)) | shoelace = -3 | cross = -3
((0, 0), (4, 0), (0, 3)) | shoelace = 12 | cross = 12
((1, 1), (2, 2), (3, 3)) | shoelace = 0 | cross = 0


Tam giác vuông `(0,0),(4,0),(0,3)` có diện tích `4·3/2 = 6` → `2S = 12` ✓.

## 3. Cách trực giác

### 3.1. Cách 1 — Kiểm tra phân biệt + so sánh hệ số góc bằng `float` (có xử lý đường đứng)

Người mới thường nghĩ: (1) 3 điểm khác nhau? (2) hệ số góc AB khác AC? Vì chia cho 0 nên phải tách trường hợp đường thẳng đứng.

In [7]:
def is_boomerang_slope_float(points) -> bool:
    (x1, y1), (x2, y2), (x3, y3) = points
    if (x1, y1) == (x2, y2) or (x1, y1) == (x3, y3) or (x2, y2) == (x3, y3):
        return False
    dx1, dy1 = x2 - x1, y2 - y1
    dx2, dy2 = x3 - x1, y3 - y1
    if dx1 == 0 or dx2 == 0:              # có đường thẳng đứng
        return not (dx1 == 0 and dx2 == 0)
    return dy1 / dx1 != dy2 / dx2

print(is_boomerang_slope_float([[1, 1], [2, 3], [3, 2]]))
print(is_boomerang_slope_float([[1, 1], [2, 2], [3, 3]]))
print(is_boomerang_slope_float([[0, 0], [0, 2], [0, 5]]))

True
False
False


Cách này **chạy đúng** với miền `0..100` (sẽ kiểm chứng ở mục 10), nhưng: nhiều nhánh, dễ quên trường hợp, và dựa vào `float`. Ta muốn một công thức **không nhánh, không chia**.

### 3.2. Cách 2 — So sánh hệ số góc bằng `Fraction` (phân số chính xác)

`fractions.Fraction(a, b)` lưu phân số **tối giản chính xác** — không sai số. Vẫn phải tách đường thẳng đứng vì `Fraction(a, 0)` báo lỗi.

In [8]:
from fractions import Fraction

print(Fraction(2, 4), Fraction(3, 6) == Fraction(1, 2), Fraction(1, 3) * 3 == 1)

def is_boomerang_fraction(points) -> bool:
    (x1, y1), (x2, y2), (x3, y3) = points
    if len({(x1, y1), (x2, y2), (x3, y3)}) < 3:
        return False
    dx1, dy1, dx2, dy2 = x2 - x1, y2 - y1, x3 - x1, y3 - y1
    if dx1 == 0 or dx2 == 0:
        return not (dx1 == 0 and dx2 == 0)
    return Fraction(dy1, dx1) != Fraction(dy2, dx2)

print(is_boomerang_fraction([[1, 1], [2, 3], [3, 2]]), is_boomerang_fraction([[0, 0], [1, 1], [1, 1]]))

1/2 True True
True False


`len({...}) < 3`: đưa 3 điểm (dạng **tuple** vì list không bỏ vào `set` được) vào một set; set tự loại trùng → nếu còn ít hơn 3 phần tử thì có điểm trùng.

In [9]:
print({(0, 0), (0, 0), (1, 1)})
try:
    {[0, 0], [1, 1]}
except TypeError as e:
    print("list không hash được:", e)

{(1, 1), (0, 0)}
list không hash được: unhashable type: 'list'


## 4. Nút thắt & insight

Hai cách trên đều cần:
- kiểm tra **trùng điểm** riêng,
- tách nhánh **đường thẳng đứng** riêng.

Insight từ gợi ý của đề: *boomerang ⟺ diện tích tam giác ≠ 0* ⟺ `cross(AB, AC) ≠ 0`.

Một biểu thức số nguyên duy nhất xử lý **mọi** trường hợp:
- Điểm trùng: nếu B = A thì AB = (0, 0) → cross = 0. Nếu B = C thì AB = AC → cross(u, u) = 0.
- Thẳng hàng (kể cả thẳng đứng/nằm ngang): AB ∥ AC → cross = 0.
- Không cần chia → không có `ZeroDivisionError`, không có sai số `float`.

In [10]:
tests = {
    "tam giác thường": [[1, 1], [2, 3], [3, 2]],
    "thẳng hàng chéo": [[1, 1], [2, 2], [3, 3]],
    "thẳng đứng": [[0, 0], [0, 2], [0, 5]],
    "nằm ngang": [[0, 7], [3, 7], [9, 7]],
    "A trùng B": [[4, 4], [4, 4], [1, 2]],
    "B trùng C": [[0, 0], [5, 5], [5, 5]],
    "cả 3 trùng": [[2, 2], [2, 2], [2, 2]],
}
for name, ((x1, y1), (x2, y2), (x3, y3)) in tests.items():
    cr = (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
    print(f"{name:<16} cross = {cr:>4} -> boomerang? {cr != 0}")

tam giác thường  cross =   -3 -> boomerang? True
thẳng hàng chéo  cross =    0 -> boomerang? False
thẳng đứng       cross =    0 -> boomerang? False
nằm ngang        cross =    0 -> boomerang? False
A trùng B        cross =    0 -> boomerang? False
B trùng C        cross =    0 -> boomerang? False
cả 3 trùng       cross =    0 -> boomerang? False


`tests.items()` cho từng cặp `(key, value)`; phần `((x1, y1), (x2, y2), (x3, y3))` unpack luôn value ngay trong vòng `for`.

## 5. Lời giải chính — tích có hướng khác 0

```python
(x1, y1), (x2, y2), (x3, y3) = points          # Đoạn 1: tách tọa độ
return (x2 - x1) * (y3 - y1) != (y2 - y1) * (x3 - x1)   # Đoạn 2: cross(AB, AC) != 0
```

Viết `a != b` thay cho `a - b != 0` — cùng nghĩa, dễ đọc như "nhân chéo hai hệ số góc **khác nhau**".

In [11]:
def is_boomerang_cross(points) -> bool:
    (x1, y1), (x2, y2), (x3, y3) = points
    return (x2 - x1) * (y3 - y1) != (y2 - y1) * (x3 - x1)

for name, pts in tests.items():
    print(f"{name:<16} -> {is_boomerang_cross(pts)}")

tam giác thường  -> True
thẳng hàng chéo  -> False
thẳng đứng       -> False
nằm ngang        -> False
A trùng B        -> False
B trùng C        -> False
cả 3 trùng       -> False


### 5.1. Cách 4 — Shoelace (cùng bản chất, viết theo tọa độ)

In [12]:
def is_boomerang_shoelace(points) -> bool:
    (x1, y1), (x2, y2), (x3, y3) = points
    return x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2) != 0

print([is_boomerang_shoelace(p) for p in tests.values()])

[True, False, False, False, False, False, False]


### 5.2. Cách 5 — Hướng chuẩn hóa bằng `gcd`

Một cách "đếm hướng" khác (rất hữu ích ở bài nhiều điểm như LeetCode 149): biểu diễn hướng AB bằng cặp `(dx, dy)` **tối giản** và **chuẩn hóa dấu**. Thẳng hàng ⟺ hướng AB và AC trùng nhau.

`math.gcd(a, b)` là ước chung lớn nhất (luôn ≥ 0).

In [13]:
from math import gcd

def normalized_direction(dx, dy):
    g = gcd(dx, dy)
    dx, dy = dx // g, dy // g
    if dx < 0 or (dx == 0 and dy < 0):     # chuẩn hóa: (−1, −2) và (1, 2) là cùng một đường
        dx, dy = -dx, -dy
    return dx, dy

def is_boomerang_gcd(points) -> bool:
    (x1, y1), (x2, y2), (x3, y3) = points
    if len({(x1, y1), (x2, y2), (x3, y3)}) < 3:
        return False
    return normalized_direction(x2 - x1, y2 - y1) != normalized_direction(x3 - x1, y3 - y1)

print(normalized_direction(4, 6), normalized_direction(-2, -3), normalized_direction(0, -5))
print([is_boomerang_gcd(p) for p in tests.values()])

(2, 3) (2, 3) (0, 1)
[True, False, False, False, False, False, False]


Vì sao cần chuẩn hóa dấu? Hướng `(2, 3)` và `(−2, −3)` ngược chiều nhưng **cùng một đường thẳng** (điểm C ở phía bên kia A). Nếu không chuẩn hóa, hai điểm hai phía của A sẽ bị coi là "không thẳng hàng".

## 6. TRACE từng bước

### 6.1. Ví dụ 1 `[[1,1],[2,3],[3,2]]`

| Bước | Biểu thức | Giá trị |
|---|---|---|
| vector AB | `(2-1, 3-1)` | `(1, 2)` |
| vector AC | `(3-1, 2-1)` | `(2, 1)` |
| `(x2-x1)(y3-y1)` | `1 · 1` | `1` |
| `(y2-y1)(x3-x1)` | `2 · 2` | `4` |
| cross | `1 - 4` | `-3` ≠ 0 → **True** |

Dấu âm: C nằm **bên phải** hướng A → B (thứ tự A, B, C quay theo chiều kim đồng hồ). Diện tích tam giác = 3/2.

### 6.2. Ví dụ 2 `[[1,1],[2,2],[3,3]]`: AB = (1,1), AC = (2,2), cross = `1·2 - 1·2 = 0` → **False**.

In [14]:
def trace_boomerang(points):
    (x1, y1), (x2, y2), (x3, y3) = points
    ab = (x2 - x1, y2 - y1)
    ac = (x3 - x1, y3 - y1)
    left = ab[0] * ac[1]
    right = ab[1] * ac[0]
    cr = left - right
    print(f"points = {points}")
    print(f"  AB = {ab}, AC = {ac}")
    print(f"  AB.x * AC.y = {ab[0]} * {ac[1]} = {left}")
    print(f"  AB.y * AC.x = {ab[1]} * {ac[0]} = {right}")
    turn = "trái (ngược kim đồng hồ)" if cr > 0 else ("phải (thuận kim đồng hồ)" if cr < 0 else "thẳng hàng / trùng")
    print(f"  cross = {cr} -> {turn} -> boomerang = {cr != 0}, diện tích = {abs(cr) / 2}")
    return cr != 0

trace_boomerang([[1, 1], [2, 3], [3, 2]])
trace_boomerang([[1, 1], [2, 2], [3, 3]])
trace_boomerang([[0, 0], [0, 2], [0, 5]])
trace_boomerang([[4, 4], [4, 4], [1, 2]])

points = [[1, 1], [2, 3], [3, 2]]
  AB = (1, 2), AC = (2, 1)
  AB.x * AC.y = 1 * 1 = 1
  AB.y * AC.x = 2 * 2 = 4
  cross = -3 -> phải (thuận kim đồng hồ) -> boomerang = True, diện tích = 1.5
points = [[1, 1], [2, 2], [3, 3]]
  AB = (1, 1), AC = (2, 2)
  AB.x * AC.y = 1 * 2 = 2
  AB.y * AC.x = 1 * 2 = 2
  cross = 0 -> thẳng hàng / trùng -> boomerang = False, diện tích = 0.0
points = [[0, 0], [0, 2], [0, 5]]
  AB = (0, 2), AC = (0, 5)
  AB.x * AC.y = 0 * 5 = 0
  AB.y * AC.x = 2 * 0 = 0
  cross = 0 -> thẳng hàng / trùng -> boomerang = False, diện tích = 0.0
points = [[4, 4], [4, 4], [1, 2]]
  AB = (0, 0), AC = (-3, -2)
  AB.x * AC.y = 0 * -2 = 0
  AB.y * AC.x = 0 * -3 = 0
  cross = 0 -> thẳng hàng / trùng -> boomerang = False, diện tích = 0.0


False

## 7. Code nộp LeetCode (Python 3)

In [15]:
class Solution:
    def isBoomerang(self, points: list[list[int]]) -> bool:
        (x1, y1), (x2, y2), (x3, y3) = points
        return (x2 - x1) * (y3 - y1) != (y2 - y1) * (x3 - x1)

sol = Solution()
assert sol.isBoomerang([[1, 1], [2, 3], [3, 2]]) is True
assert sol.isBoomerang([[1, 1], [2, 2], [3, 3]]) is False
print("Solution PASS 2 ví dụ của đề")

Solution PASS 2 ví dụ của đề


## 8. Vì sao đúng?

Đặt `u = AB`, `v = AC`. Ta chứng minh: **A, B, C là boomerang ⟺ cross(u, v) ≠ 0**.

**(⇐) Nếu cross ≠ 0:**
- `u ≠ (0,0)` và `v ≠ (0,0)` (nếu một vector bằng 0 thì cross = 0) → B ≠ A, C ≠ A.
- `u ≠ v` (vì cross(u, u) = 0) → B ≠ C.
- `u` không song song `v` (song song ⟺ `v = t·u` ⟹ cross = t·cross(u,u) = 0) → C không nằm trên đường AB.
→ 3 điểm phân biệt, không thẳng hàng.

**(⇒) Nếu boomerang:** giả sử cross = 0 với `u ≠ 0`. Khi đó `ux·vy = uy·vx` → `v` tỉ lệ với `u` → C nằm trên đường thẳng qua A theo hướng `u`, tức A, B, C thẳng hàng — mâu thuẫn. Vậy cross ≠ 0.

Ý nghĩa hình học: `|cross|/2` là diện tích tam giác ABC; diện tích bằng 0 ⟺ tam giác "suy biến" (bẹp thành đoạn thẳng hoặc điểm).

**Không tràn số:** tọa độ ≤ 100 → mỗi tích ≤ `100·100 = 10^4`. Python không giới hạn số nguyên nên cũng không lo trong mọi trường hợp.

## 9. Phân tích Time / Space

Input cố định 3 điểm → mọi cách đều `O(1)` thời gian và bộ nhớ. Khác nhau ở **độ an toàn và độ gọn**:

| Cách | Ý tưởng | Nhánh đặc biệt | Rủi ro | Time / Space |
|---|---|---|---|---|
| 1. Slope `float` | so `dy/dx` | trùng điểm, đường đứng | sai số `float` (nếu tọa độ lớn) | `O(1)` / `O(1)` |
| 2. `Fraction` | phân số chính xác | trùng điểm, đường đứng | chậm hơn chút | `O(1)` / `O(1)` |
| 3. Cross product | `cross(AB, AC) ≠ 0` | **không có** | không | `O(1)` / `O(1)` |
| 4. Shoelace | diện tích ≠ 0 | không có | không | `O(1)` / `O(1)` |
| 5. Hướng `gcd` | so hướng tối giản | trùng điểm | quên chuẩn hóa dấu | `O(log V)` (gcd) / `O(1)` |

(`V` là độ lớn tọa độ; `gcd` theo thuật toán Euclid tốn `O(log V)`.)

## 10. Kiểm thử: mẫu đề, biên, đối chiếu **toàn bộ** miền nhỏ

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

cases = [
    ("đề 1", [[1, 1], [2, 3], [3, 2]], True),
    ("đề 2", [[1, 1], [2, 2], [3, 3]], False),
    ("thẳng đứng", [[0, 0], [0, 2], [0, 5]], False),
    ("nằm ngang", [[0, 7], [3, 7], [9, 7]], False),
    ("A trùng B", [[4, 4], [4, 4], [1, 2]], False),
    ("A trùng C", [[4, 4], [1, 2], [4, 4]], False),
    ("cả 3 trùng", [[2, 2], [2, 2], [2, 2]], False),
    ("C ở phía bên kia A", [[5, 5], [6, 7], [4, 3]], False),
    ("góc vuông", [[0, 0], [0, 100], [100, 0]], True),
    ("gần thẳng hàng", [[0, 0], [99, 100], [100, 101]], True),
]
for name, pts, exp in cases:
    check(name, Solution().isBoomerang(pts), exp)

đề 1                         PASS (True)
đề 2                         PASS (False)
thẳng đứng                   PASS (False)
nằm ngang                    PASS (False)
A trùng B                    PASS (False)
A trùng C                    PASS (False)
cả 3 trùng                   PASS (False)
C ở phía bên kia A           PASS (False)
góc vuông                    PASS (True)
gần thẳng hàng               PASS (True)


Ca "C ở phía bên kia A": AB = (1,2), AC = (−1,−2) — ngược chiều nhưng cùng đường → không phải boomerang. Ca "gần thẳng hàng": độ dốc `100/99` và `101/100` rất gần nhau nhưng khác → boomerang.

### 10.1. Đối chiếu mọi cách trên toàn bộ bộ ba điểm trong lưới 0..4

Lưới 5×5 có 25 điểm → `25³ = 15 625` bộ ba (có thứ tự, kể cả trùng). Đủ nhỏ để thử **hết**.

In [17]:
from itertools import product

grid = [(x, y) for x in range(5) for y in range(5)]
solvers = {
    "slope_float": is_boomerang_slope_float,
    "fraction": is_boomerang_fraction,
    "cross": is_boomerang_cross,
    "shoelace": is_boomerang_shoelace,
    "gcd": is_boomerang_gcd,
}
count_true = 0
for p1, p2, p3 in product(grid, repeat=3):
    pts = [list(p1), list(p2), list(p3)]
    expected = Solution().isBoomerang(pts)
    count_true += expected
    for name, f in solvers.items():
        assert f(pts) == expected, (name, pts)
print(f"{25 ** 3} bộ ba: mọi cách khớp nhau ({count_true} boomerang)")

15625 bộ ba: mọi cách khớp nhau (12888 boomerang)


### 10.2. Tính chất: đổi thứ tự 3 điểm không đổi kết quả

Boomerang là tính chất của **tập** 3 điểm. Hoán vị thứ tự chỉ có thể đổi **dấu** của cross, không đổi việc nó khác 0 hay không.

In [18]:
from itertools import permutations
import random
random.seed(1037)
for _ in range(2000):
    pts = [[random.randint(0, 6), random.randint(0, 6)] for _ in range(3)]
    results = {Solution().isBoomerang(list(p)) for p in permutations(pts)}
    assert len(results) == 1
print("Kết quả không phụ thuộc thứ tự điểm (2000 bộ ngẫu nhiên × 6 hoán vị)")

Kết quả không phụ thuộc thứ tự điểm (2000 bộ ngẫu nhiên × 6 hoán vị)


`permutations(pts)` sinh mọi cách sắp xếp 3 phần tử (3! = 6). Đưa kết quả vào `set` — nếu chỉ còn 1 phần tử thì mọi hoán vị cho cùng đáp án.

## 11. Sai lầm người mới hay mắc

### 11.1. Chỉ kiểm tra thẳng hàng bằng slope mà quên đường thẳng đứng

In [19]:
def wrong_no_vertical(points):
    (x1, y1), (x2, y2), (x3, y3) = points
    return (y2 - y1) / (x2 - x1) != (y3 - y1) / (x3 - x1)

try:
    print(wrong_no_vertical([[0, 0], [0, 2], [1, 5]]))
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)
print("đúng:", Solution().isBoomerang([[0, 0], [0, 2], [1, 5]]))

ZeroDivisionError: division by zero
đúng: True


**Vì sao sai:** AB thẳng đứng → `x2 - x1 = 0` → chia cho 0, chương trình **crash**. **Sửa:** nhân chéo.

### 11.2. Chỉ kiểm tra "3 điểm phân biệt" hoặc chỉ kiểm tra "không thẳng hàng" bằng công thức không bắt được trùng điểm

In [20]:
def wrong_only_distinct(points):
    return len({tuple(p) for p in points}) == 3          # SAI: quên thẳng hàng

def wrong_slope_no_dup(points):
    (x1, y1), (x2, y2), (x3, y3) = points
    s1 = (y2 - y1) / (x2 - x1) if x2 != x1 else float("inf")
    s2 = (y3 - y1) / (x3 - x1) if x3 != x1 else float("inf")
    return s1 != s2                                        # SAI: A trùng B -> s1 = inf (coi như đường đứng)

print("chỉ phân biệt:", wrong_only_distinct([[1, 1], [2, 2], [3, 3]]), "| đúng:", Solution().isBoomerang([[1, 1], [2, 2], [3, 3]]))
print("slope không xét trùng:", wrong_slope_no_dup([[4, 4], [4, 4], [1, 2]]), "| đúng:", Solution().isBoomerang([[4, 4], [4, 4], [1, 2]]))

chỉ phân biệt: True | đúng: False
slope không xét trùng: True | đúng: False


**Vì sao sai:** cần **cả hai** điều kiện. Ở hàm thứ hai, A trùng B làm `x2 == x1` nên bị coi là "đường thẳng đứng" với hệ số góc vô cực — khác hệ số góc AC → trả `True` sai. `float("inf")` là giá trị vô cực của Python. **Sửa:** dùng cross product (tự bắt cả hai) hoặc kiểm tra trùng trước.

### 11.3. Chuẩn hóa hướng bằng `gcd` nhưng quên chuẩn hóa dấu

In [21]:
def wrong_gcd_no_sign(points):
    (x1, y1), (x2, y2), (x3, y3) = points
    if len({(x1, y1), (x2, y2), (x3, y3)}) < 3:
        return False
    def norm(dx, dy):
        g = gcd(dx, dy)
        return dx // g, dy // g
    return norm(x2 - x1, y2 - y1) != norm(x3 - x1, y3 - y1)

print("sai :", wrong_gcd_no_sign([[5, 5], [6, 7], [4, 3]]))
print("đúng:", Solution().isBoomerang([[5, 5], [6, 7], [4, 3]]))

sai : True
đúng: False


**Vì sao sai:** AB = (1, 2), AC = (−1, −2) → hướng tối giản khác dấu nhưng **cùng đường thẳng**. **Sửa:** chuẩn hóa dấu như `normalized_direction`, hoặc dùng cross product.

### 11.4. Dùng khoảng cách và bất đẳng thức tam giác với `sqrt`

Ý tưởng "thẳng hàng ⟺ `AB + BC == AC` (với B ở giữa)" đúng về toán, nhưng `sqrt` cho `float` → so sánh `==` có thể sai do làm tròn, và còn phải xét điểm nào ở giữa. Không nên dùng khi đã có công thức số nguyên.

In [22]:
from math import dist
for A_, B_, C_ in [((0, 0), (1, 3), (2, 6)), ((0, 0), (1, 1), (3, 3))]:   # cả hai bộ đều thẳng hàng, B ở giữa
    lhs, rhs = dist(A_, B_) + dist(B_, C_), dist(A_, C_)
    print(f"{A_, B_, C_}: AB+BC = {lhs!r}, AC = {rhs!r}, bằng nhau? {lhs == rhs}")

((0, 0), (1, 3), (2, 6)): AB+BC = 6.324555320336759, AC = 6.324555320336759, bằng nhau? True
((0, 0), (1, 1), (3, 3)): AB+BC = 4.242640687119286, AC = 4.242640687119285, bằng nhau? False


`math.dist(p, q)` là khoảng cách Euclid. Bộ đầu tình cờ bằng nhau, nhưng bộ `(0,0),(1,1),(3,3)` — rõ ràng thẳng hàng — lại cho `AB + BC` lệch `AC` ở chữ số thập phân cuối do làm tròn → so sánh `==` báo "không thẳng hàng" sai. So sánh `==` với `float` không đáng tin.

## 12. Nhận diện pattern

Dấu hiệu:
- "3 điểm thẳng hàng?", "diện tích tam giác", "điểm nằm bên trái/phải đường thẳng", "hướng quay" → **cross product 2D**.
- Tọa độ nguyên → luôn ưu tiên **nhân chéo**, tránh chia và `float`.
- Nhiều điểm cùng đường thẳng → **hướng tối giản bằng gcd** làm khóa của `dict`.

Bài liên quan:
- LeetCode 1232 (Check If It Is a Straight Line) — n điểm thẳng hàng.
- LeetCode 149 (Max Points on a Line) — hướng chuẩn hóa + hash map.
- LeetCode 812 (Largest Triangle Area) — shoelace.
- LeetCode 587 (Erect the Fence) — bao lồi, dùng dấu cross product.

## 13. Biến thể

### 13.1. Biến thể A — hướng quay của 3 điểm (orientation)

In [23]:
def orientation(a, b2, c) -> str:
    cr = (b2[0] - a[0]) * (c[1] - a[1]) - (b2[1] - a[1]) * (c[0] - a[0])
    if cr > 0:
        return "ngược kim đồng hồ (C bên trái A→B)"
    if cr < 0:
        return "thuận kim đồng hồ (C bên phải A→B)"
    return "thẳng hàng"

print(orientation((0, 0), (4, 0), (2, 3)))
print(orientation((0, 0), (4, 0), (2, -3)))
print(orientation((0, 0), (4, 0), (9, 0)))

ngược kim đồng hồ (C bên trái A→B)
thuận kim đồng hồ (C bên phải A→B)
thẳng hàng


### 13.2. Biến thể B — diện tích tam giác (trả về số thực)

In [24]:
def triangle_area(points) -> float:
    (x1, y1), (x2, y2), (x3, y3) = points
    return abs((x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)) / 2

print(triangle_area([[0, 0], [4, 0], [0, 3]]), triangle_area([[1, 1], [2, 3], [3, 2]]))

6.0 1.5


## 14. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `[[2,1],[5,3],[8,5]]`: tính AB, AC, cross. Có phải boomerang không?

### Bài 2 — Dự đoán output
Đoán `normalized_direction(-6, 4)`, `normalized_direction(0, 0)` (cẩn thận!) và `triangle_area([[0,0],[2,0],[0,2]])`.

### Bài 3 — TODO: viết lại lời giải cross product

In [25]:
def exercise_is_boomerang(points: list[list[int]]) -> bool:
    # TODO: unpack 3 điểm, tính cross(AB, AC), so sánh với 0
    pass

# assert exercise_is_boomerang([[1, 1], [2, 3], [3, 2]]) is True
# assert exercise_is_boomerang([[0, 0], [0, 2], [0, 5]]) is False
# assert exercise_is_boomerang([[4, 4], [4, 4], [1, 2]]) is False

### Bài 4 — TODO: n điểm có cùng nằm trên một đường thẳng không? (LeetCode 1232, các điểm phân biệt, `n ≥ 2`)

In [26]:
def exercise_check_straight_line(coordinates: list[list[int]]) -> bool:
    # TODO: so mọi điểm với hai điểm đầu bằng cross product
    pass

# assert exercise_check_straight_line([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]) is True
# assert exercise_check_straight_line([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]) is False
# assert exercise_check_straight_line([[0, 0], [0, 1], [0, -1]]) is True

### Bài 5 — TODO biến thể: đếm số bộ ba (không tính thứ tự) trong danh sách điểm tạo thành boomerang

In [27]:
def exercise_count_boomerang_triples(pts: list[list[int]]) -> int:
    # TODO: dùng itertools.combinations(pts, 3)
    pass

# assert exercise_count_boomerang_triples([[0, 0], [1, 1], [2, 2], [0, 1]]) == 3
# assert exercise_count_boomerang_triples([[0, 0], [1, 1], [2, 2]]) == 0

### Bài 6 — Mini challenge: điểm `P` có nằm **trong** tam giác ABC (kể cả trên cạnh) không?

Gợi ý: dùng dấu của 3 cross product `cross(AB, AP)`, `cross(BC, BP)`, `cross(CA, CP)`.

In [28]:
def exercise_point_in_triangle(a, b2, c, p) -> bool:
    # TODO
    pass

# assert exercise_point_in_triangle((0, 0), (4, 0), (0, 4), (1, 1)) is True
# assert exercise_point_in_triangle((0, 0), (4, 0), (0, 4), (2, 2)) is True     # trên cạnh
# assert exercise_point_in_triangle((0, 0), (4, 0), (0, 4), (3, 3)) is False
# assert exercise_point_in_triangle((0, 0), (0, 4), (4, 0), (1, 1)) is True     # thứ tự đỉnh đảo chiều

## 15. Gợi ý ba mức

**Bài 1** — (1) AB = (3, 2); (2) AC = (6, 4); (3) `3·4 - 2·6`.

**Bài 2** — (1) gcd(−6, 4) = 2; (2) chuẩn hóa để `dx > 0`; (3) `gcd(0, 0) = 0` → phép chia `// 0` sẽ lỗi — đó là lý do `is_boomerang_gcd` loại điểm trùng **trước**.

**Bài 3** — (1) mục 5; (2) `(x2-x1)*(y3-y1)`; (3) `!= (y2-y1)*(x3-x1)`.

**Bài 4** — (1) cố định `A = coordinates[0]`, `B = coordinates[1]`; (2) với mỗi điểm `P`, cross(AB, AP) phải bằng 0; (3) `all(...)`.

**Bài 5** — (1) `from itertools import combinations`; (2) với mỗi bộ ba gọi `Solution().isBoomerang`; (3) `sum(1 for t in combinations(...) if ...)`.

**Bài 6** — (1) tính 3 giá trị `d1, d2, d3`; (2) P ở trong (hoặc trên cạnh) ⟺ ba giá trị **không** có cả dấu âm lẫn dấu dương; (3) `has_neg = d1 < 0 or d2 < 0 or d3 < 0`, tương tự `has_pos`, trả `not (has_neg and has_pos)`.

## 16. Đáp án tham khảo

**Bài 1:** AB = (3,2), AC = (6,4), cross = `12 - 12 = 0` → **không** phải boomerang (cùng đường `y = (2x - 1)/3`).

**Bài 2:** `(−6, 4)` → chia gcd 2 → `(−3, 2)` → đổi dấu → `(3, −2)`. `normalized_direction(0, 0)` → `ZeroDivisionError`. Diện tích = `2.0`.

In [29]:
from itertools import combinations


def answer_is_boomerang(points):
    (x1, y1), (x2, y2), (x3, y3) = points
    return (x2 - x1) * (y3 - y1) != (y2 - y1) * (x3 - x1)


def answer_check_straight_line(coordinates):
    (x0, y0), (x1, y1) = coordinates[0], coordinates[1]
    dx, dy = x1 - x0, y1 - y0
    return all(dx * (y - y0) == dy * (x - x0) for x, y in coordinates)


def answer_count_boomerang_triples(pts):
    return sum(1 for t in combinations(pts, 3) if answer_is_boomerang(list(t)))


def answer_point_in_triangle(a, b2, c, p):
    def cr(o, u, v):
        return (u[0] - o[0]) * (v[1] - o[1]) - (u[1] - o[1]) * (v[0] - o[0])
    d1, d2, d3 = cr(a, b2, p), cr(b2, c, p), cr(c, a, p)
    has_neg = d1 < 0 or d2 < 0 or d3 < 0
    has_pos = d1 > 0 or d2 > 0 or d3 > 0
    return not (has_neg and has_pos)


assert answer_is_boomerang([[2, 1], [5, 3], [8, 5]]) is False
assert normalized_direction(-6, 4) == (3, -2)
try:
    normalized_direction(0, 0)
    raise AssertionError("phải lỗi")
except ZeroDivisionError:
    pass
assert triangle_area([[0, 0], [2, 0], [0, 2]]) == 2.0
assert answer_is_boomerang([[1, 1], [2, 3], [3, 2]]) is True
assert answer_is_boomerang([[0, 0], [0, 2], [0, 5]]) is False
assert answer_is_boomerang([[4, 4], [4, 4], [1, 2]]) is False
assert answer_check_straight_line([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]) is True
assert answer_check_straight_line([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]) is False
assert answer_check_straight_line([[0, 0], [0, 1], [0, -1]]) is True
assert answer_count_boomerang_triples([[0, 0], [1, 1], [2, 2], [0, 1]]) == 3
assert answer_count_boomerang_triples([[0, 0], [1, 1], [2, 2]]) == 0
assert answer_point_in_triangle((0, 0), (4, 0), (0, 4), (1, 1)) is True
assert answer_point_in_triangle((0, 0), (4, 0), (0, 4), (2, 2)) is True
assert answer_point_in_triangle((0, 0), (4, 0), (0, 4), (3, 3)) is False
assert answer_point_in_triangle((0, 0), (0, 4), (4, 0), (1, 1)) is True

# kiểm tra point_in_triangle bằng diện tích: P trong tam giác ⟺ S(PAB)+S(PBC)+S(PCA) == S(ABC)
random.seed(5)
for _ in range(3000):
    a, b2, c, p = [(random.randint(0, 6), random.randint(0, 6)) for _ in range(4)]
    if not answer_is_boomerang([list(a), list(b2), list(c)]):
        continue
    twice = lambda u, v, w: abs((v[0] - u[0]) * (w[1] - u[1]) - (v[1] - u[1]) * (w[0] - u[0]))
    inside = twice(p, a, b2) + twice(p, b2, c) + twice(p, c, a) == twice(a, b2, c)
    assert answer_point_in_triangle(a, b2, c, p) == inside
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 6:** `cross(AB, AP)` dương nghĩa là P ở bên trái cạnh A→B. P nằm trong tam giác ⟺ P ở **cùng một phía** của cả 3 cạnh (đi vòng A→B→C→A). Nếu có giá trị bằng 0, P nằm trên đường chứa cạnh đó — ta vẫn coi là "trong" nếu hai giá trị còn lại không mâu thuẫn dấu. Cách viết `has_neg`/`has_pos` không phụ thuộc chiều đánh số đỉnh.

Kiểm chứng bằng diện tích: P trong tam giác ⟺ tổng diện tích 3 tam giác con bằng diện tích tam giác lớn (dùng `2S` nguyên để tránh `float`).

## 17. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def isBoomerang(self, points: list[list[int]]) -> bool:
        (x1, y1), (x2, y2), (x3, y3) = points
        return (x2 - x1) * (y3 - y1) != (y2 - y1) * (x3 - x1)
```

**Câu thần chú:** *"Boomerang ⟺ diện tích ≠ 0 ⟺ cross(AB, AC) ≠ 0 — nhân chéo, đừng chia."*

**Checklist trước khi submit:**
- [ ] Không chia (tránh `ZeroDivisionError` với đường thẳng đứng)?
- [ ] Không dùng `float` để so sánh bằng?
- [ ] Điểm trùng nhau được xử lý (cross product tự lo)?
- [ ] Đã test: thẳng đứng, nằm ngang, trùng điểm, C ở phía bên kia A?
- [ ] Trả về `bool` (`!=` cho ra `True/False`)?